In [1]:
import os

# CLEAN ALL conflicting Spark env vars
os.environ.pop("SPARK_HOME", None)
os.environ.pop("PYSPARK_SUBMIT_ARGS", None)
os.environ.pop("PYSPARK_PYTHON", None)
os.environ.pop("PYSPARK_DRIVER_PYTHON", None)

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Taxi Demand Modeling") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print("Spark initialized successfully")

26/04/06 15:53:33 WARN Utils: Your hostname, Prashants-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en0)
26/04/06 15:53:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 15:53:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark initialized successfully


In [2]:
train_df = spark.read.parquet("/Users/prashant/Desktop/Projects/CRN-Taxi/yellow_taxi/yellow_taxi/model_input/train.parquet")
val_df   = spark.read.parquet("/Users/prashant/Desktop/Projects/CRN-Taxi/yellow_taxi/yellow_taxi/model_input/val.parquet")
test_df  = spark.read.parquet("/Users/prashant/Desktop/Projects/CRN-Taxi/yellow_taxi/yellow_taxi/model_input/test.parquet")

print("Data loaded successfully")

Data loaded successfully


In [3]:
print("Train:", train_df.count())
print("Val:", val_df.count())
print("Test:", test_df.count())

train_df.printSchema()

train_df.select(
    "lat_bin", "lon_bin", "timestamp", "demand", "lag_1", "rolling_3"
).show(5, truncate=False)

Train: 2213197
Val: 524262
Test: 654811
root
 |-- lat_bin: long (nullable = true)
 |-- lon_bin: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- demand: long (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- long_trip_ratio: double (nullable = true)
 |-- lag_1: long (nullable = true)
 |-- lag_2: long (nullable = true)
 |-- lag_3: long (nullable = true)
 |-- rolling_3: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- hour_sin: double (nullable = true)
 |-- hour_cos: double (nullable = true)
 |-- loc_avg_demand: double (nullable = true)
 |-- loc_std_demand: double (nullable = true)
 |-- is_peak_hour: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_night: integer (nullable = true)
 |-- demand_vs_avg:

In [4]:
from pyspark.ml.feature import VectorAssembler

# Drop leakage / unsafe columns
drop_cols = [
    "timestamp", "pickup_date",
    "has_demand",
    "demand_vs_avg", "demand_zscore",
    "loc_avg_demand", "loc_std_demand",
    "log_demand"
]

train_clean = train_df.drop(*drop_cols)
val_clean   = val_df.drop(*drop_cols)
test_clean  = test_df.drop(*drop_cols)

# Define target
target_col = "demand"

# Feature columns
feature_cols = [c for c in train_clean.columns if c != target_col]

# Assemble features
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

train_final = assembler.transform(train_clean).select("features", target_col)
val_final   = assembler.transform(val_clean).select("features", target_col)
test_final  = assembler.transform(test_clean).select("features", target_col)

print("Feature vector created")

Feature vector created


In [5]:
train_final.show(5)

26/04/06 15:53:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+------+
|            features|demand|
+--------------------+------+
|(31,[0,1,2,3,4,5,...|     1|
|(31,[0,1,2,3,4,5,...|     1|
|(31,[0,1,2,3,4,5,...|     1|
|(31,[0,1,2,3,4,10...|     1|
|(31,[0,1,2,3,4,5,...|     1|
+--------------------+------+
only showing top 5 rows



In [6]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="demand",
    predictionCol="prediction",
    maxIter=50,
    regParam=0.1,
    elasticNetParam=0.0  # pure ridge (stable for many features)
)

lr_model = lr.fit(train_final)

print("Model trained")

26/04/06 15:53:37 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/06 15:53:38 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Model trained


In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate(df, name):
    preds = lr_model.transform(df)

    evaluator_rmse = RegressionEvaluator(labelCol="demand", predictionCol="prediction", metricName="rmse")
    evaluator_r2   = RegressionEvaluator(labelCol="demand", predictionCol="prediction", metricName="r2")
    evaluator_mse  = RegressionEvaluator(labelCol="demand", predictionCol="prediction", metricName="mse")

    rmse = evaluator_rmse.evaluate(preds)
    r2   = evaluator_r2.evaluate(preds)
    mse  = evaluator_mse.evaluate(preds)

    print(f"\n{name} Metrics:")
    print(f"RMSE: {rmse}")
    print(f"MSE : {mse}")
    print(f"R2  : {r2}")

    return preds

train_preds = evaluate(train_final, "Train")
val_preds   = evaluate(val_final, "Validation")


Train Metrics:
RMSE: 40.18270006440268
MSE : 1614.6493844657473
R2  : 0.9735461065421807

Validation Metrics:
RMSE: 49.492500625613424
MSE : 2449.507618176345
R2  : 0.9441704037016516


In [8]:
from pyspark.sql.functions import col, count, when, avg, stddev, min, max

train_df.select(
    count("*").alias("total_rows"),
    count(when(col("demand") == 0, True)).alias("zero_demand"),
    (count(when(col("demand") == 0, True)) / count("*")).alias("zero_ratio"),
    avg("demand").alias("mean_demand"),
    stddev("demand").alias("std_demand"),
    min("demand").alias("min_demand"),
    max("demand").alias("max_demand")
).show()

+----------+-----------+------------------+------------------+-----------------+----------+----------+
|total_rows|zero_demand|        zero_ratio|       mean_demand|       std_demand|min_demand|max_demand|
+----------+-----------+------------------+------------------+-----------------+----------+----------+
|   2213197|    2117041|0.9565533479396547|4.4187783554740045|55.51321369622848|         0|      2658|
+----------+-----------+------------------+------------------+-----------------+----------+----------+



In [9]:
train_df.approxQuantile(
    "demand",
    [0.5, 0.75, 0.9, 0.95, 0.99],
    0.01
)

[0.0, 0.0, 0.0, 0.0, 2658.0]

In [10]:
train_df.select("demand", "has_demand").show(10)

+------+----------+
|demand|has_demand|
+------+----------+
|     0|         0|
|     0|         0|
|     0|         0|
|     0|         0|
|     0|         0|
|     0|         0|
|     0|         0|
|     0|         0|
|     0|         0|
|     0|         0|
+------+----------+
only showing top 10 rows



In [11]:
train_df.selectExpr(
    "sum(case when demand > 0 then 1 else 0 end) as actual_positive",
    "sum(has_demand) as has_demand_sum"
).show()

+---------------+--------------+
|actual_positive|has_demand_sum|
+---------------+--------------+
|          96156|         96156|
+---------------+--------------+



In [12]:
train_reg = train_final.filter("demand > 0")
val_reg   = val_final.filter("demand > 0")
test_reg  = test_final.filter("demand > 0")

print("Train (active):", train_reg.count())
print("Val (active):", val_reg.count())
print("Test (active):", test_reg.count())

Train (active): 96156
Val (active): 92941
Test (active): 277978


In [13]:
from pyspark.ml.regression import LinearRegression

lr_active = LinearRegression(
    featuresCol="features",
    labelCol="demand",
    predictionCol="prediction",
    maxIter=50,
    regParam=0.1,
    elasticNetParam=0.0
)

lr_active_model = lr_active.fit(train_reg)

print("Active demand model trained")

Active demand model trained


In [14]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Train fresh model ONLY on active data
lr_active = LinearRegression(
    featuresCol="features",
    labelCol="demand",
    predictionCol="prediction",
    maxIter=50,
    regParam=0.1,
    elasticNetParam=0.0
)

lr_active_model = lr_active.fit(train_reg)

print("Model retrained on ACTIVE data")

# Evaluation function
def evaluate(df, name):
    preds = lr_active_model.transform(df)

    evaluator_rmse = RegressionEvaluator(labelCol="demand", predictionCol="prediction", metricName="rmse")
    evaluator_r2   = RegressionEvaluator(labelCol="demand", predictionCol="prediction", metricName="r2")

    rmse = evaluator_rmse.evaluate(preds)
    r2   = evaluator_r2.evaluate(preds)

    print(f"\n{name}:")
    print(f"RMSE: {rmse}")
    print(f"R2  : {r2}")

evaluate(train_reg, "Train (Active)")
evaluate(val_reg, "Validation (Active)")

Model retrained on ACTIVE data

Train (Active):
RMSE: 40.18270006440268
R2  : 0.9735461065421807

Validation (Active):
RMSE: 49.492500625613424
R2  : 0.9441704037016516


In [15]:
train_reg.selectExpr(
    "count(*) as total",
    "sum(case when demand = 0 then 1 else 0 end) as zero_count"
).show()

+-----+----------+
|total|zero_count|
+-----+----------+
|96156|         0|
+-----+----------+



In [16]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="demand",
    predictionCol="prediction",
    maxIter=50,        # number of trees
    maxDepth=6,        # controls complexity
    stepSize=0.1,      # learning rate
    subsamplingRate=0.8,
    seed=42
)

gbt_model = gbt.fit(train_reg)

print("GBT model trained")

GBT model trained


In [17]:
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate(df, name):
    preds = gbt_model.transform(df)

    evaluator_rmse = RegressionEvaluator(labelCol="demand", predictionCol="prediction", metricName="rmse")
    evaluator_r2   = RegressionEvaluator(labelCol="demand", predictionCol="prediction", metricName="r2")

    rmse = evaluator_rmse.evaluate(preds)
    r2   = evaluator_r2.evaluate(preds)

    print(f"\n{name}:")
    print(f"RMSE: {rmse}")
    print(f"R2  : {r2}")

    return preds

train_preds_gbt = evaluate(train_reg, "Train (GBT)")
val_preds_gbt   = evaluate(val_reg, "Validation (GBT)")


Train (GBT):
RMSE: 28.698899435874054
R2  : 0.9865059643359723

Validation (GBT):
RMSE: 44.13661656932371
R2  : 0.9555999183462998


In [18]:
import pandas as pd

# Extract importance
importances = gbt_model.featureImportances.toArray()

# Map to feature names
fi_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances
}).sort_values(by="importance", ascending=False)

fi_df.head(15)

,feature,importance
6,lag_1,0.616826
25,log_lag_1,0.084353
20,lag_168,0.056584
13,hour_sin,0.022466
11,day_of_week,0.022304
21,trend_1_3,0.021524
2,pickup_hour,0.019862
1,lon_bin,0.018727
19,lag_24,0.018002
18,rolling_6,0.015171


In [19]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="demand",
    predictionCol="prediction",
    numTrees=50,
    maxDepth=8,
    seed=42
)

rf_model = rf.fit(train_reg)

print("Random Forest trained")

26/04/06 15:54:06 WARN DAGScheduler: Broadcasting large task binary with size 1176.1 KiB
26/04/06 15:54:07 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB


Random Forest trained


In [20]:
def evaluate_model(model, df, name):
    preds = model.transform(df)

    evaluator_rmse = RegressionEvaluator(labelCol="demand", predictionCol="prediction", metricName="rmse")
    evaluator_r2   = RegressionEvaluator(labelCol="demand", predictionCol="prediction", metricName="r2")

    rmse = evaluator_rmse.evaluate(preds)
    r2   = evaluator_r2.evaluate(preds)

    print(f"\n{name}:")
    print(f"RMSE: {rmse}")
    print(f"R2  : {r2}")

In [21]:
evaluate_model(rf_model, train_reg, "Train (RF)")
evaluate_model(rf_model, val_reg, "Validation (RF)")


Train (RF):
RMSE: 38.20971758486943
R2  : 0.9760801184760033

Validation (RF):
RMSE: 43.988962760709555
R2  : 0.9558964918389548


In [22]:
from pyspark.sql.functions import avg, count

zone_df = train_df.groupBy("lat_bin", "lon_bin").agg(
    avg("demand").alias("avg_demand"),
    count("*").alias("activity"),
    avg("avg_fare").alias("avg_fare"),
    avg("avg_distance").alias("avg_distance")
)

In [23]:
zone_df.head()

Row(lat_bin=4063, lon_bin=-7402, avg_demand=0.02600780234070221, activity=2307, avg_fare=15.788461538461538, avg_distance=5.4513461538461545)

In [24]:
from pyspark.sql.functions import isnan, col, sum as spark_sum

zone_df.select([
    spark_sum(isnan(col(c)).cast("int")).alias(c)
    for c in ["avg_demand", "activity", "avg_fare", "avg_distance"]
]).show()

+----------+--------+--------+------------+
|avg_demand|activity|avg_fare|avg_distance|
+----------+--------+--------+------------+
|         0|       0|       0|           0|
+----------+--------+--------+------------+



In [25]:
from pyspark.sql.functions import isnan, when

zone_df_clean = zone_df

for c in ["avg_demand", "activity", "avg_fare", "avg_distance"]:
    zone_df_clean = zone_df_clean.withColumn(
        c,
        when(col(c).isNull() | isnan(col(c)), 0).otherwise(col(c))
    )

In [26]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

assembler = VectorAssembler(
    inputCols=["avg_demand", "activity", "avg_fare", "avg_distance"],
    outputCol="features"
)

zone_vec = assembler.transform(zone_df_clean)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(zone_vec)
zone_scaled = scaler_model.transform(zone_vec)

In [27]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(
    k=5,
    featuresCol="scaled_features",
    seed=42
)

kmeans_model = kmeans.fit(zone_scaled)

zone_clustered = kmeans_model.transform(zone_scaled)

In [28]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="scaled_features",
    predictionCol="prediction"
)

for k in range(3, 8):
    kmeans = KMeans(k=k, featuresCol="scaled_features", seed=42)
    model = kmeans.fit(zone_scaled)
    preds = model.transform(zone_scaled)

    silhouette = evaluator.evaluate(preds)

    print(f"K={k}, Silhouette={silhouette}")

K=3, Silhouette=0.7443270043645487
K=4, Silhouette=0.7140278640052999
K=5, Silhouette=0.6909635841982663
K=6, Silhouette=0.6815600090974882
K=7, Silhouette=0.6825539400003715


In [29]:
from pyspark.sql.functions import avg, count

zone_clustered.groupBy("prediction").agg(
    count("*").alias("num_zones"),
    avg("avg_demand").alias("avg_demand"),
    avg("activity").alias("avg_activity"),
    avg("avg_fare").alias("avg_fare"),
    avg("avg_distance").alias("avg_distance")
).orderBy("avg_demand", ascending=False).show(20, False)

+----------+---------+--------------------+------------------+------------------+-------------------+
|prediction|num_zones|avg_demand          |avg_activity      |avg_fare          |avg_distance       |
+----------+---------+--------------------+------------------+------------------+-------------------+
|1         |23       |122.86240652543296  |2658.8695652173915|10.939918368864396|2.5254708214583608 |
|3         |125      |6.276899263718854   |2598.784          |13.751758985535098|3.4723724452415032 |
|2         |76       |0.8258168002524028  |2324.434210526316 |49.08484683896198 |12.401282881895224 |
|0         |408      |0.025766278430984595|2292.9387254901962|17.467895030139655|4.268383309669484  |
|4         |316      |4.190848132618735E-4|2262.71835443038  |1.2123099537268671|0.23891004856281584|
+----------+---------+--------------------+------------------+------------------+-------------------+



In [30]:
zone_clustered.groupBy("prediction").count().orderBy("count", ascending=False).show()

+----------+-----+
|prediction|count|
+----------+-----+
|         0|  408|
|         4|  316|
|         3|  125|
|         2|   76|
|         1|   23|
+----------+-----+



In [31]:
zone_clustered.filter("prediction = 0").show(5)

+-------+-------+--------------------+--------+------------------+------------------+--------------------+--------------------+----------+
|lat_bin|lon_bin|          avg_demand|activity|          avg_fare|      avg_distance|            features|     scaled_features|prediction|
+-------+-------+--------------------+--------+------------------+------------------+--------------------+--------------------+----------+
|   4063|  -7402| 0.02600780234070221|    2307|15.788461538461538|5.4513461538461545|[0.02600780234070...|[-0.1899435888843...|         0|
|   4064|  -7400|  0.0170082860880942|    2293|14.337837837837839|3.5143243243243236|[0.01700828608809...|[-0.1903864484739...|         0|
|   4068|  -7392| 0.10595446584938704|    2284|14.709287148594377|3.7054487951807222|[0.10595446584938...|[-0.1860094722496...|         0|
|   4084|  -7386|0.014486391571553995|    2278|11.441964285714286|3.0370535714285714|[0.01448639157155...|[-0.1905105490521...|         0|
|   4070|  -7385|0.00177698

In [32]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="scaled_features",
    predictionCol="prediction",
    metricName="silhouette"
)

silhouette = evaluator.evaluate(zone_clustered)

print("Silhouette Score:", silhouette)
print("WSSSE:", kmeans_model.summary.trainingCost)

Silhouette Score: 0.6909635841982663
WSSSE: 687.2349695192546


In [33]:
from pyspark.sql.functions import col

zone_clustered_final = zone_clustered.select(
    "lat_bin", "lon_bin",
    col("prediction").alias("zone_id")
)

train_zone = train_df.join(zone_clustered_final, ["lat_bin", "lon_bin"], "left")
val_zone   = val_df.join(zone_clustered_final, ["lat_bin", "lon_bin"], "left")
test_zone  = test_df.join(zone_clustered_final, ["lat_bin", "lon_bin"], "left")

print("Zone ID added to datasets")

Zone ID added to datasets


In [34]:
test_zone.columns 

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand',
 'weight',
 'zone_id']

In [35]:
train_zone.write.mode("overwrite").parquet("model_input/train_with_zone.parquet")
val_zone.write.mode("overwrite").parquet("model_input/val_with_zone.parquet")
test_zone.write.mode("overwrite").parquet("model_input/test_with_zone.parquet")

print("Datasets with zone_id saved")

Datasets with zone_id saved


In [36]:
kmeans_model.write().overwrite().save("models/kmeans_zone_model")

In [37]:
train_zone_log = train_zone.withColumn("target", col("log_demand"))
val_zone_log   = val_zone.withColumn("target", col("log_demand"))
test_zone_log  = test_zone.withColumn("target", col("log_demand"))

feature_cols_zone = feature_cols + ["zone_id"]

assembler = VectorAssembler(
    inputCols=feature_cols_zone,
    outputCol="features",
    handleInvalid="skip"
)

train_vec = assembler.transform(train_zone_log).select("features", "target")
val_vec   = assembler.transform(val_zone_log).select("features", "target")

gbt_log = GBTRegressor(
    featuresCol="features",
    labelCol="target",
    maxIter=50,
    maxDepth=6,
    stepSize=0.1,
    seed=42
)

gbt_log_model = gbt_log.fit(train_vec)

In [38]:
from pyspark.sql.functions import exp

preds = gbt_log_model.transform(val_vec)

preds = preds.withColumn(
    "prediction_actual",
    exp(col("prediction")) - 1
)

In [39]:
from pyspark.sql.functions import exp

preds = gbt_log_model.transform(val_vec)

preds = preds.withColumn(
    "prediction_actual",
    exp(col("prediction")) - 1
)

preds = preds.withColumn(
    "actual_demand",
    exp(col("target")) - 1
)

In [40]:
from pyspark.sql.functions import exp, col

val_preds = gbt_log_model.transform(val_vec)

val_preds = val_preds.withColumn(
    "prediction_actual",
    exp(col("prediction")) - 1
).withColumn(
    "actual_demand",
    exp(col("target")) - 1
)

from pyspark.sql.functions import avg, abs, when

metrics = val_preds.select(
    avg((col("actual_demand") - col("prediction_actual"))**2).alias("mse"),
    avg(abs(col("actual_demand") - col("prediction_actual"))).alias("mae"),
    avg(
        when(col("actual_demand") != 0,
             abs((col("actual_demand") - col("prediction_actual")) / col("actual_demand"))
        )
    ).alias("mape")
).collect()[0]

rmse = metrics["mse"] ** 0.5

print("\nValidation (GBT Log + Zone):")
print(f"RMSE: {rmse}")
print(f"MAE : {metrics['mae']}")
print(f"MAPE: {metrics['mape']}")


Validation (GBT Log + Zone):
RMSE: 86.14238374547703
MAE : 24.648714988557632
MAPE: 0.35885998387325063


In [41]:
val_preds.select(
    "actual_demand",
    "prediction_actual"
).show(10)



+-------------+------------------+
|actual_demand| prediction_actual|
+-------------+------------------+
|          1.0|0.9545450693886295|
|          1.0| 1.076244405486169|
|          1.0|0.9364178907488816|
|          1.0|1.0319030599995882|
|          1.0|1.0401575879100031|
|          1.0|0.8973122904958724|
|          1.0|0.9643987148216413|
|          1.0|0.9932101969193339|
|          1.0|1.0789042218538727|
|          1.0|1.0283246322884039|
+-------------+------------------+
only showing top 10 rows



In [42]:
from pyspark.ml.feature import VectorAssembler

# IMPORTANT: same feature list used before
feature_cols_zone = feature_cols + ["zone_id"]

assembler_zone = VectorAssembler(
    inputCols=feature_cols_zone,
    outputCol="features",
    handleInvalid="skip"
)

train_zone_vec = assembler_zone.transform(train_zone).select("features", "demand")
val_zone_vec   = assembler_zone.transform(val_zone).select("features", "demand")

print("Feature vectors rebuilt")

from pyspark.ml.regression import GBTRegressor

gbt_zone = GBTRegressor(
    featuresCol="features",
    labelCol="demand",
    maxIter=50,
    maxDepth=6,
    stepSize=0.1,
    seed=42
)

gbt_zone_model = gbt_zone.fit(train_zone_vec)

print("Raw GBT model ready")

from pyspark.sql.functions import col

val_preds_raw = gbt_zone_model.transform(val_zone_vec)

val_preds_raw = val_preds_raw.select(
    col("prediction").alias("pred_raw"),
    col("demand").alias("actual_demand")
)

val_preds_raw.show(5)


Feature vectors rebuilt


Raw GBT model ready
+------------------+-------------+
|          pred_raw|actual_demand|
+------------------+-------------+
|1.4570141771689975|            1|
|1.1068603040882046|            1|
|1.9939830736805384|            1|
|1.3782660820971426|            1|
|1.2438688668519846|            1|
+------------------+-------------+
only showing top 5 rows



In [43]:
val_base = val_zone.select("*")
val_raw = gbt_zone_model.transform(val_zone_vec)

val_raw = val_raw.withColumnRenamed("prediction", "pred_raw")

from pyspark.sql.functions import exp

val_log = gbt_log_model.transform(val_vec)

val_log = val_log.withColumn(
    "pred_log",
    exp(col("prediction")) - 1
)

In [44]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

w = Window.orderBy(col("actual_demand"))

# Raw
val_preds_raw_idx = val_preds_raw.withColumn(
    "row_id", row_number().over(w)
)

# Log
val_preds_log_idx = val_preds.withColumn(
    "row_id", row_number().over(w)
).withColumnRenamed("prediction_actual", "pred_log")

In [45]:
val_final = val_preds_raw_idx.join(
    val_preds_log_idx,
    on="row_id",
    how="inner"
).select(
    val_preds_raw_idx["actual_demand"],
    "pred_raw",
    "pred_log"
)

# Hybrid
alpha = 0.5

val_final = val_final.withColumn(
    "pred_hybrid",
    col("pred_log") + alpha * (col("pred_raw") - col("pred_log"))
)

val_final.show(10)

26/04/06 15:54:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1

+-------------+------------------+------------------+------------------+
|actual_demand|          pred_raw|          pred_log|       pred_hybrid|
+-------------+------------------+------------------+------------------+
|            1|1.4570141771689975|0.9545450693886295|1.2057796232788136|
|            1|1.1068603040882046| 1.076244405486169|1.0915523547871868|
|            1|1.9939830736805384|0.9364178907488816|  1.46520048221471|
|            1|1.3782660820971426|1.0319030599995882|1.2050845710483653|
|            1|1.2438688668519846|1.0401575879100031| 1.142013227380994|
|            1|1.4623765340504167|0.8973122904958724|1.1798444122731446|
|            1|1.4039604104844154|0.9643987148216413|1.1841795626530285|
|            1|1.9037265082580854|0.9932101969193339|1.4484683525887097|
|            1| 2.513810718965517|1.0789042218538727| 1.796357470409695|
|            1|1.4570141771689975|1.0283246322884039|1.2426694047287006|
+-------------+------------------+-----------------

26/04/06 15:54:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [46]:
from pyspark.sql.functions import avg, abs, when

metrics = val_final.select(
    avg((col("actual_demand") - col("pred_hybrid"))**2).alias("mse"),
    avg(abs(col("actual_demand") - col("pred_hybrid"))).alias("mae"),
    avg(
        when(col("actual_demand") != 0,
             abs((col("actual_demand") - col("pred_hybrid")) / col("actual_demand"))
        )
    ).alias("mape")
).collect()[0]

rmse = metrics["mse"] ** 0.5

print("\nHybrid Model:")
print(f"RMSE: {rmse}")
print(f"MAE : {metrics['mae']}")
print(f"MAPE: {metrics['mape']}")

26/04/06 15:54:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1


Hybrid Model:
RMSE: 58.99346928671435
MAE : 18.850097678125703
MAPE: 0.4479638077417804


26/04/06 15:54:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [47]:
from pyspark.sql.functions import when, col

val_final = val_final.withColumn(
    "pred_smart",
    when(col("pred_raw") > 10, col("pred_raw"))  # spikes → raw
    .otherwise(col("pred_log"))                 # normal → log
)

In [48]:
metrics = val_final.select(
    avg((col("actual_demand") - col("pred_smart"))**2).alias("mse"),
    avg(abs(col("actual_demand") - col("pred_smart"))).alias("mae"),
    avg(
        when(col("actual_demand") != 0,
             abs((col("actual_demand") - col("pred_smart")) / col("actual_demand"))
        )
    ).alias("mape")
).collect()[0]

rmse = metrics["mse"] ** 0.5

print("\nSMART Hybrid:")
print(f"RMSE: {rmse}")
print(f"MAE : {metrics['mae']}")
print(f"MAPE: {metrics['mape']}")

26/04/06 15:54:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1


SMART Hybrid:
RMSE: 44.90332164310524
MAE : 15.580343029899826
MAPE: 0.4025649374384051


In [49]:
from pyspark.sql.functions import col

val_final = val_final.withColumn(
    "weight",
    col("pred_raw") / (col("pred_raw") + 5)   # smooth transition
)

val_final = val_final.withColumn(
    "pred_smart2",
    col("weight") * col("pred_raw") +
    (1 - col("weight")) * col("pred_log")
)

In [50]:
from pyspark.sql.functions import col, avg, abs, when

def evaluate(df, pred_col):
    
    metrics = df.select(
        avg((col("actual_demand") - col(pred_col))**2).alias("mse"),
        avg(abs(col("actual_demand") - col(pred_col))).alias("mae"),
        avg(
            when(col("actual_demand") != 0,
                 abs((col("actual_demand") - col(pred_col)) / col("actual_demand"))
            )
        ).alias("mape")
    ).collect()[0]

    rmse = metrics["mse"] ** 0.5

    print(f"\nMetrics for {pred_col}:")
    print(f"RMSE: {rmse}")
    print(f"MAE : {metrics['mae']}")
    print(f"MAPE: {metrics['mape']}")

In [51]:
evaluate(val_final, "pred_raw")
evaluate(val_final, "pred_log")
evaluate(val_final, "pred_smart")
evaluate(val_final, "pred_smart2")

26/04/06 15:54:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1


Metrics for pred_raw:
RMSE: 45.043024412226124
MAE : 15.940852767437258
MAPE: 0.5895762743913356


26/04/06 15:54:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1


Metrics for pred_log:
RMSE: 86.14238374547696
MAE : 24.648714988557327
MAPE: 0.35885998387325146


26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1


Metrics for pred_smart:
RMSE: 44.90332164310524
MAE : 15.580343029899826
MAPE: 0.4025649374384051


26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1


Metrics for pred_smart2:
RMSE: 45.46361973310004
MAE : 15.64055965119743
MAPE: 0.4742463903315027


26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1

In [52]:
thresholds = [3, 5, 10, 20, 30]

for t in thresholds:
    temp = val_final.withColumn(
        "pred_test",
        when(col("pred_raw") > t, col("pred_raw"))
        .otherwise(col("pred_log"))
    )
    
    print(f"\nThreshold = {t}")
    evaluate(temp, "pred_test")

26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.



Threshold = 3


26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:54:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1


Metrics for pred_test:
RMSE: 44.94445841991206
MAE : 15.773616200630372
MAPE: 0.5053593641572021

Threshold = 5


26/04/06 15:55:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1

...

26/04/06 15:55:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1


Metrics for pred_test:
RMSE: 44.80966342936717
MAE : 15.410402423052874
MAPE: 0.3799029360336932

Threshold = 30


26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1


Metrics for pred_test:
RMSE: 44.67054712957987
MAE : 15.251850875055508
MAPE: 0.3694652416864761


26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1

In [53]:
from pyspark.sql.functions import when, col, count

val_final.withColumn(
    "model_used",
    when(col("pred_raw") > 30, "raw").otherwise("log")
).groupBy("model_used").count().show()

26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 1

+----------+-----+
|model_used|count|
+----------+-----+
|       log|70288|
|       raw|22653|
+----------+-----+



26/04/06 15:55:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/06 15:55:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [54]:
test_zone_vec = assembler.transform(test_zone)
test_pred_raw = gbt_zone_model.transform(test_zone_vec)
from pyspark.sql.functions import exp

test_pred_log = gbt_log_model.transform(test_zone_vec) \
    .withColumn("pred_log", exp(col("prediction")) - 1)



In [55]:
from pyspark.sql.functions import col, exp, when

# RAW
test_pred_raw = gbt_zone_model.transform(test_zone_vec).select(
    "lat_bin", "lon_bin", "timestamp",
    col("prediction").alias("pred_raw"),
    col("demand").alias("actual_demand")
)

# LOG
test_pred_log = gbt_log_model.transform(test_zone_vec).select(
    "lat_bin", "lon_bin", "timestamp",
    (exp(col("prediction")) - 1).alias("pred_log")
)

# ✅ SAFE JOIN
test_final = test_pred_raw.join(
    test_pred_log,
    on=["lat_bin", "lon_bin", "timestamp"],
    how="inner"
)

# ✅ HYBRID LOGIC
test_final = test_final.withColumn(
    "pred_final",
    when(col("pred_raw") > 30, col("pred_raw"))
    .otherwise(col("pred_log"))
)

# ✅ EVALUATE
evaluate(test_final, "pred_final")


Metrics for pred_final:
RMSE: 36.93679403187482
MAE : 12.612500600763525
MAPE: 0.33485024110352424


In [56]:
from pyspark.sql.functions import col, mean, pow

# Step 1 — mean of actual demand
mean_actual = test_final.select(mean(col("actual_demand"))).collect()[0][0]

# Step 2 — compute SSE and SST
r2_df = test_final.select(
    pow(col("actual_demand") - col("pred_final"), 2).alias("sq_error"),
    pow(col("actual_demand") - mean_actual, 2).alias("sq_total")
)

agg_vals = r2_df.agg(
    {"sq_error": "sum", "sq_total": "sum"}
).collect()[0]

SSE = agg_vals["sum(sq_error)"]
SST = agg_vals["sum(sq_total)"]

# Step 3 — R²
r2 = 1 - (SSE / SST)

print("R2:", r2)

R2: 0.9721175779563216


In [57]:
gbt_zone_model.write().overwrite().save("yellow_taxi/models/gbt_raw_model")
gbt_log_model.write().overwrite().save("yellow_taxi/models/gbt_log_model")
kmeans_model.write().overwrite().save("yellow_taxi/models/kmeans_zone_model")

assembler.write().overwrite().save("yellow_taxi/models/assembler_raw")

scaler_model.write().overwrite().save("yellow_taxi/models/scaler_model")
# zone_assembler is not defined, so this line is removed


In [58]:
import json

config = {
    "threshold": 30,
    "models": {
        "raw": "gbt_raw_model",
        "log": "gbt_log_model",
        "cluster": "kmeans_zone_model"
    }
}

with open("yellow_taxi/models/config.json", "w") as f:
    json.dump(config, f)

In [59]:
from pyspark.sql.functions import col

# Join zone_id from test_zone into test_final
final_output = test_final.join(
    test_zone.select("lat_bin", "lon_bin", "timestamp", "zone_id"),
    on=["lat_bin", "lon_bin", "timestamp"],
    how="left"
).select(
    col("lat_bin"),
    col("lon_bin"),
    col("zone_id"),
    col("timestamp"),
    col("actual_demand"),
    col("pred_raw"),
    col("pred_log"),
    col("pred_final")
)

In [60]:
test_final.count()

277978

In [61]:
final_output.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("yellow_taxi/output/predictions_csv_test")

In [62]:
zone_summary = final_output.groupBy("zone_id").agg(
    {"actual_demand": "sum", "pred_final": "sum"}
).withColumnRenamed("sum(actual_demand)", "actual_total") \
 .withColumnRenamed("sum(pred_final)", "pred_total") \
 .orderBy(col("pred_total").desc())

zone_summary.show()

+-------+------------+-------------------+
|zone_id|actual_total|         pred_total|
+-------+------------+-------------------+
|      1|    19347072|1.993769713417478E7|
|      3|     5618001|   5624635.22544731|
|      2|      531667|  518742.6711714403|
|      0|       67927|  65932.44130277816|
|      4|        5521|  4931.661676209748|
+-------+------------+-------------------+



In [63]:
from pyspark.sql.functions import hour

time_summary = final_output.withColumn("hour", hour("timestamp")) \
    .groupBy("hour") \
    .agg({"pred_final": "sum"}) \
    .orderBy("hour")

time_summary.show()

+----+------------------+
|hour|   sum(pred_final)|
+----+------------------+
|   0| 923971.0810674614|
|   1| 661016.7364035775|
|   2|476813.19318211486|
|   3| 340945.9185592711|
|   4|257065.29998025412|
|   5| 272067.3045792523|
|   6| 585749.3586504294|
|   7|1025278.8854964091|
|   8|1243823.8800909652|
|   9|1244591.9827168195|
|  10|1175236.7020552806|
|  11|1183261.0661657862|
|  12|1253388.6896864243|
|  13|1259744.5105923363|
|  14|1286010.5830411513|
|  15|1279058.6220018726|
|  16| 1168058.280118763|
|  17|1336392.7802210243|
|  18|1651209.5011212206|
|  19| 1716012.416507946|
+----+------------------+
only showing top 20 rows



In [64]:
top_zones = zone_summary.limit(10)
top_zones.show()

+-------+------------+-------------------+
|zone_id|actual_total|         pred_total|
+-------+------------+-------------------+
|      1|    19347072|1.993769713417478E7|
|      3|     5618001|   5624635.22544731|
|      2|      531667|  518742.6711714403|
|      0|       67927|  65932.44130277816|
|      4|        5521|  4931.661676209748|
+-------+------------+-------------------+



In [65]:
df_model = spark.read.parquet('/Users/prashant/Desktop/Projects/CRN-Taxi/yellow_taxi/yellow_taxi/data/df_model_full')

In [66]:
print("Rows:", df_model.count())
print("Columns:", len(df_model.columns))

df_model.printSchema()
df_model.show(5)

Rows: 10209012
Columns: 39
root
 |-- lat_bin: long (nullable = true)
 |-- lon_bin: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- demand: long (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- long_trip_ratio: double (nullable = true)
 |-- lag_1: long (nullable = true)
 |-- lag_2: long (nullable = true)
 |-- lag_3: long (nullable = true)
 |-- rolling_3: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- hour_sin: double (nullable = true)
 |-- hour_cos: double (nullable = true)
 |-- loc_avg_demand: double (nullable = true)
 |-- loc_std_demand: double (nullable = true)
 |-- is_peak_hour: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_night: integer (nullable = true)
 |-- demand_vs_avg: double (nullable = true)
 |-- demand_zscore: double (

In [67]:
from pyspark.sql.functions import col

df_model_clean = df_model.fillna({
    "pickup_hour": 0,
    "avg_fare": 0.0,
    "avg_distance": 0.0,
    "long_trip_ratio": 0.0,
    "lag_1": 0,
    "lag_2": 0,
    "lag_3": 0,
    "rolling_3": 0.0,
    "loc_avg_demand": 0.0,
    "loc_std_demand": 0.0,
    "demand_vs_avg": 0.0,
    "demand_zscore": 0.0,
    "rolling_6": 0.0,
    "lag_24": 0,
    "lag_168": 0,
    "trend_1_3": 0.0,
    "trend_1_6": 0.0,
    "log_lag_1": 0.0,
    "log_rolling_6": 0.0,
    "recent_activity": 0.0,
    "log_demand": 0.0,
    "persistence": 0
})

from pyspark.sql.functions import sum, col

df_model_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_model_clean.columns
]).show()

+-------+-------+---------+-----------+------+--------+------------+---------------+-----+-----+-----+---------+----+-----------+-----+--------+--------+--------------+--------------+------------+----------+--------+-------------+-------------+---------+------+-------+---------+---------+--------------+--------------+---------+-------------+-----------+---------------+----------+-----------+----------+-----------+
|lat_bin|lon_bin|timestamp|pickup_hour|demand|avg_fare|avg_distance|long_trip_ratio|lag_1|lag_2|lag_3|rolling_3|hour|day_of_week|month|hour_sin|hour_cos|loc_avg_demand|loc_std_demand|is_peak_hour|is_weekend|is_night|demand_vs_avg|demand_zscore|rolling_6|lag_24|lag_168|trend_1_3|trend_1_6|is_active_zone|peak_intensity|log_lag_1|log_rolling_6|hour_x_zone|recent_activity|log_demand|persistence|has_demand|pickup_date|
+-------+-------+---------+-----------+------+--------+------------+---------------+-----+-----+-----+---------+----+-----------+-----+--------+--------+-----------

In [68]:
df_model_clean.describe().show()

+-------+------------------+------------------+------------------+-----------------+-----------------+-------------------+--------------------+-----------------+-----------------+-----------------+------------------+------------------+------------------+-----------------+--------------------+--------------------+--------------------+-------------------+------------------+------------------+-------------------+------------------+--------------------+------------------+-----------------+-----------------+--------------------+--------------------+-------------------+------------------+-------------------+-------------------+-----------------+------------------+-------------------+-------------------+-------------------+
|summary|           lat_bin|           lon_bin|       pickup_hour|           demand|         avg_fare|       avg_distance|     long_trip_ratio|            lag_1|            lag_2|            lag_3|         rolling_3|              hour|       day_of_week|            month| 

In [69]:
from pyspark.sql.functions import col

df_model_clean.groupBy("demand").count().orderBy("demand").show(10)

+------+-------+
|demand|  count|
+------+-------+
|     0|9741937|
|     1| 130142|
|     2|  48482|
|     3|  29232|
|     4|  20296|
|     5|  14854|
|     6|  11859|
|     7|   9706|
|     8|   8441|
|     9|   7047|
+------+-------+
only showing top 10 rows



In [70]:
total = df_model_clean.count()

zero_count = df_model_clean.filter(col("demand") == 0).count()
non_zero_count = df_model_clean.filter(col("demand") > 0).count()

print("Total:", total)
print("Zero demand:", zero_count)
print("Non-zero demand:", non_zero_count)
print("Zero %:", zero_count / total)

Total: 10209012
Zero demand: 9741937
Non-zero demand: 467075
Zero %: 0.9542487559031178


In [71]:
df_model_clean.select(
    "demand",
    "avg_fare",
    "avg_distance",
    "lag_1",
    "rolling_3"
).summary().show()

+-------+-----------------+-----------------+-------------------+-----------------+------------------+
|summary|           demand|         avg_fare|       avg_distance|            lag_1|         rolling_3|
+-------+-----------------+-----------------+-------------------+-----------------+------------------+
|  count|         10209012|         10209012|           10209012|         10209012|          10209012|
|   mean|4.249123519494344|0.749242136709545|0.19016478443434567|4.248197670842194| 4.246094039266554|
| stddev|51.81090923903298|4.578496926773919| 1.2260164674041123|51.80055297675804| 51.02849772421838|
|    min|                0|              0.0|                0.0|                0|               0.0|
|    25%|                0|              0.0|                0.0|                0|               0.0|
|    50%|                0|              0.0|                0.0|                0|               0.0|
|    75%|                0|              0.0|                0.0|        

In [72]:
df_model_clean.sample(0.0001).show(20, truncate=False)

+-------+-------+-------------------+-----------+------+--------+------------+---------------+-----+-----+-----+---------+----+-----------+-----+----------------------+-----------------------+---------------------+--------------------+------------+----------+--------+-------------+---------------------+---------+------+-------+---------+---------+--------------+---------------------+---------+-------------+---------------------+---------------+----------+-----------+----------+-----------+
|lat_bin|lon_bin|timestamp          |pickup_hour|demand|avg_fare|avg_distance|long_trip_ratio|lag_1|lag_2|lag_3|rolling_3|hour|day_of_week|month|hour_sin              |hour_cos               |loc_avg_demand       |loc_std_demand      |is_peak_hour|is_weekend|is_night|demand_vs_avg|demand_zscore        |rolling_6|lag_24|lag_168|trend_1_3|trend_1_6|is_active_zone|peak_intensity       |log_lag_1|log_rolling_6|hour_x_zone          |recent_activity|log_demand|persistence|has_demand|pickup_date|
+-------+-

In [73]:
print(kmeans_model.getFeaturesCol())

scaled_features


In [74]:
print(scaler_model.getInputCol())
print(scaler_model.getOutputCol())

features
scaled_features


In [75]:
print(assembler.getOutputCol())

features


In [76]:
zone_map = zone_clustered_final.select("lat_bin", "lon_bin", "zone_id").dropDuplicates()

df_with_zone = df_model.join(
    zone_map,
    on=["lat_bin", "lon_bin"],
    how="left"
)


df_with_zone.select("zone_id").distinct().count()

df_with_zone.filter(col("zone_id").isNull()).count()

0

In [77]:
print("Before:", df_model.count())
print("After :", df_with_zone.count())

Before: 10209012
After : 10209012


In [78]:
df_with_zone.groupBy("lat_bin", "lon_bin").count() \
    .filter(col("count") > 1).count()

948

In [79]:
df_with_zone.groupBy("zone_id").count().orderBy("zone_id").show()

+-------+-------+
|zone_id|  count|
+-------+-------+
|      0|4393752|
|      1| 247687|
|      2| 818444|
|      3|1346125|
|      4|3403004|
+-------+-------+



In [80]:
from pyspark.sql.functions import countDistinct

df_with_zone.groupBy("lat_bin", "lon_bin") \
    .agg(countDistinct("zone_id").alias("zone_count")) \
    .filter(col("zone_count") > 1).count()

0

In [81]:
df_with_zone.select("lat_bin", "lon_bin", "zone_id").show(20)

+-------+-------+-------+
|lat_bin|lon_bin|zone_id|
+-------+-------+-------+
|   4051|  -7401|      4|
|   4089|  -7391|      0|
|   4051|  -7401|      4|
|   4089|  -7391|      0|
|   4051|  -7401|      4|
|   4089|  -7391|      0|
|   4051|  -7401|      4|
|   4089|  -7391|      0|
|   4051|  -7401|      4|
|   4089|  -7391|      0|
|   4051|  -7401|      4|
|   4089|  -7391|      0|
|   4051|  -7401|      4|
|   4089|  -7391|      0|
|   4051|  -7401|      4|
|   4089|  -7391|      0|
|   4051|  -7401|      4|
|   4089|  -7391|      0|
|   4051|  -7401|      4|
|   4089|  -7391|      0|
+-------+-------+-------+
only showing top 20 rows



In [82]:
print("weight" in df_with_zone.columns)

False


In [83]:
df_with_zone.groupBy("zone_id").agg(
    avg("lat_bin").alias("avg_lat"),
    avg("lon_bin").alias("avg_lon"),
    count("*").alias("count")
).orderBy("zone_id").show()

+-------+------------------+-------------------+-------+
|zone_id|           avg_lat|            avg_lon|  count|
+-------+------------------+-------------------+-------+
|      0| 4072.127450980392| -7390.786764705882|4393752|
|      1|4074.8695652173915| -7398.608695652174| 247687|
|      2|4069.5789473684213| -7386.355263157895| 818444|
|      3|          4073.776|          -7395.208|1346125|
|      4|4071.1455696202534|-7391.2405063291135|3403004|
+-------+------------------+-------------------+-------+



In [84]:
df_with_zone.groupBy("zone_id").agg(
    stddev("lat_bin").alias("lat_spread"),
    stddev("lon_bin").alias("lon_spread")
).show()

+-------+------------------+------------------+
|zone_id|        lat_spread|        lon_spread|
+-------+------------------+------------------+
|      1|1.8250551504407442|1.5530962742930656|
|      3| 4.984560407643356| 3.750832767698363|
|      4| 10.42807097341458|15.322249822329667|
|      2| 6.943882950993883|11.561858976710583|
|      0| 8.678260468615957| 9.183645998526906|
+-------+------------------+------------------+



In [85]:
train_clean.select("weight").summary().show()

+-------+-------------------+
|summary|             weight|
+-------+-------------------+
|  count|            2213197|
|   mean| 1.0434466520603454|
| stddev|0.20386039159908584|
|    min|                1.0|
|    25%|                1.0|
|    50%|                1.0|
|    75%|                1.0|
|    max|                2.0|
+-------+-------------------+



In [86]:
train_clean.groupBy("weight").count().show()

+------+-------+
|weight|  count|
+------+-------+
|   1.0|2117041|
|   2.0|  96156|
+------+-------+



In [87]:
print(train_clean.columns)
print("has_demand" in df_model.columns)
print("has_demand" in df_model_clean.columns)

['lat_bin', 'lon_bin', 'pickup_hour', 'demand', 'avg_fare', 'avg_distance', 'long_trip_ratio', 'lag_1', 'lag_2', 'lag_3', 'rolling_3', 'hour', 'day_of_week', 'month', 'hour_sin', 'hour_cos', 'is_peak_hour', 'is_weekend', 'is_night', 'rolling_6', 'lag_24', 'lag_168', 'trend_1_3', 'trend_1_6', 'is_active_zone', 'peak_intensity', 'log_lag_1', 'log_rolling_6', 'hour_x_zone', 'recent_activity', 'persistence', 'weight']
True
True


In [88]:
from pyspark.sql.functions import col, when

df_with_zone = df_with_zone.withColumn(
    "has_demand",
    when(col("demand") > 0, 1).otherwise(0)
)

df_with_zone = df_with_zone.withColumn(
    "weight",
    when(col("has_demand") == 1, 2.0).otherwise(1.0)
)

df_with_zone.groupBy("weight").count().show()

+------+-------+
|weight|  count|
+------+-------+
|   1.0|9741937|
|   2.0| 467075|
+------+-------+



In [89]:
missing_cols = [c for c in assembler.getInputCols() if c not in df_with_zone.columns]
print(missing_cols)

[]


In [90]:
from pyspark.sql.functions import col, sum

null_counts = df_with_zone.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in assembler.getInputCols()
])

null_counts.show()

+-------+-------+-----------+--------+------------+---------------+-----+-----+-----+---------+----+-----------+-----+--------+--------+------------+----------+--------+---------+------+-------+---------+---------+--------------+--------------+---------+-------------+-----------+---------------+-----------+------+-------+
|lat_bin|lon_bin|pickup_hour|avg_fare|avg_distance|long_trip_ratio|lag_1|lag_2|lag_3|rolling_3|hour|day_of_week|month|hour_sin|hour_cos|is_peak_hour|is_weekend|is_night|rolling_6|lag_24|lag_168|trend_1_3|trend_1_6|is_active_zone|peak_intensity|log_lag_1|log_rolling_6|hour_x_zone|recent_activity|persistence|weight|zone_id|
+-------+-------+-----------+--------+------------+---------------+-----+-----+-----+---------+----+-----------+-----+--------+--------+------------+----------+--------+---------+------+-------+---------+---------+--------------+--------------+---------+-------------+-----------+---------------+-----------+------+-------+
|      0|      0|    9741937

In [91]:
fill_cols = assembler.getInputCols()

df_filled = df_with_zone.fillna(0, subset=fill_cols)

df_filled.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in fill_cols
]).show()

+-------+-------+-----------+--------+------------+---------------+-----+-----+-----+---------+----+-----------+-----+--------+--------+------------+----------+--------+---------+------+-------+---------+---------+--------------+--------------+---------+-------------+-----------+---------------+-----------+------+-------+
|lat_bin|lon_bin|pickup_hour|avg_fare|avg_distance|long_trip_ratio|lag_1|lag_2|lag_3|rolling_3|hour|day_of_week|month|hour_sin|hour_cos|is_peak_hour|is_weekend|is_night|rolling_6|lag_24|lag_168|trend_1_3|trend_1_6|is_active_zone|peak_intensity|log_lag_1|log_rolling_6|hour_x_zone|recent_activity|persistence|weight|zone_id|
+-------+-------+-----------+--------+------------+---------------+-----+-----+-----+---------+----+-----------+-----+--------+--------+------------+----------+--------+---------+------+-------+---------+---------+--------------+--------------+---------+-------------+-----------+---------------+-----------+------+-------+
|      0|      0|          0

In [93]:
 from pyspark.ml.regression import GBTRegressionModel

df_vec = assembler.transform(df_filled)
df_scaled = scaler_model.transform(df_vec)

gbt_raw_model = GBTRegressionModel.load("yellow_taxi/models/gbt_raw_model")
gbt_log_model = GBTRegressionModel.load("yellow_taxi/models/gbt_log_model")

df_pred_raw = gbt_raw_model.transform(df_scaled)
df_pred_log = gbt_log_model.transform(df_scaled)


In [94]:
print(df_pred_raw.count(), df_pred_log.count())

10209012 10209012


In [95]:
df_pred_raw = df_pred_raw.select(
    "lat_bin", "lon_bin", "timestamp",
    col("prediction").alias("pred_raw")
)

df_pred_log = df_pred_log.select(
    "lat_bin", "lon_bin", "timestamp",
    col("prediction").alias("pred_log")
)

df_final = df_pred_raw.join(
    df_pred_log,
    on=["lat_bin", "lon_bin", "timestamp"],
    how="inner"
)

In [96]:
correct_features = [c for c in assembler.getInputCols() if c not in ["weight", "zone_id"]]

from pyspark.ml.feature import VectorAssembler

assembler_correct = VectorAssembler(
    inputCols=correct_features,
    outputCol="features"
)

df_vec = assembler_correct.transform(df_filled)
df_scaled = scaler_model.transform(df_vec)



In [97]:
df_scaled.count()

10209012

In [98]:
from pyspark.ml.regression import GBTRegressionModel

gbt_raw_model = GBTRegressionModel.load("yellow_taxi/models/gbt_raw_model")
gbt_log_model = GBTRegressionModel.load("yellow_taxi/models/gbt_log_model")

df_pred_raw = gbt_raw_model.transform(df_scaled)
df_pred_log = gbt_log_model.transform(df_scaled)

In [99]:
from pyspark.sql.functions import col

df_pred_raw.select(col("prediction").alias("pred_raw")).limit(10).show()
df_pred_log.select(col("prediction").alias("pred_log")).limit(10).show()


+------------------+
|          pred_raw|
+------------------+
|1.1304844027912053|
|1.0402278373687524|
|1.1304844027912053|
|1.1433095676282254|
|1.1304844027912053|
|1.3250534370959222|
|1.1304844027912053|
|1.1433095676282254|
|1.1304844027912053|
|1.0402278373687524|
+------------------+

+------------------+
|          pred_log|
+------------------+
|0.6791169678495236|
|0.6717823069648304|
|0.6791169678495236|
| 0.677050449349216|
|0.6689257812702168|
| 0.677050449349216|
|0.6689257812702168|
|0.6779544792024773|
|0.6689257812702168|
|0.6722517078549628|
+------------------+



In [100]:
from pyspark.sql.functions import exp

df_pred_log = df_pred_log.withColumn(
    "pred_log_exp",
    exp(col("prediction"))
)

In [101]:
df_pred_raw = df_pred_raw.select(
    "lat_bin", "lon_bin", "timestamp",
    col("prediction").alias("pred_raw")
)

df_pred_log = df_pred_log.select(
    "lat_bin", "lon_bin", "timestamp",
    col("pred_log_exp").alias("pred_log")
)

In [102]:
df_final = df_pred_raw.join(
    df_pred_log,
    on=["lat_bin", "lon_bin", "timestamp"],
    how="inner"
)

In [103]:
from pyspark.sql.functions import when

df_final = df_final.withColumn(
    "final_prediction",
    when(col("pred_raw") > 5, col("pred_raw"))  # high demand
    .otherwise(col("pred_log"))                 # low demand
)

In [104]:
print(gbt_raw_model.numFeatures)

32


In [105]:
df_vec.selectExpr("features").limit(1).show(truncate=False)


+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                               |
+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|(30,[0,1,10,11,12,13,14,15,24,27],[4051.0,-7401.0,7.0,5.0,1.0,0.9659258262890683,-0.25881904510252063,1.0,4.5716375605741976E-4,0.0032001462924019382])|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+



In [106]:
print(train_clean.columns)

['lat_bin', 'lon_bin', 'pickup_hour', 'demand', 'avg_fare', 'avg_distance', 'long_trip_ratio', 'lag_1', 'lag_2', 'lag_3', 'rolling_3', 'hour', 'day_of_week', 'month', 'hour_sin', 'hour_cos', 'is_peak_hour', 'is_weekend', 'is_night', 'rolling_6', 'lag_24', 'lag_168', 'trend_1_3', 'trend_1_6', 'is_active_zone', 'peak_intensity', 'log_lag_1', 'log_rolling_6', 'hour_x_zone', 'recent_activity', 'persistence', 'weight']


In [107]:
print("loc_avg_demand" in df_with_zone.columns)
print("loc_std_demand" in df_with_zone.columns)

True
True


In [108]:
correct_features = [
    'lat_bin', 'lon_bin', 'pickup_hour', 'avg_fare', 'avg_distance',
    'long_trip_ratio', 'lag_1', 'lag_2', 'lag_3', 'rolling_3',
    'hour', 'day_of_week', 'month', 'hour_sin', 'hour_cos',
    'is_peak_hour', 'is_weekend', 'is_night', 'rolling_6',
    'lag_24', 'lag_168', 'trend_1_3', 'trend_1_6',
    'is_active_zone', 'peak_intensity', 'log_lag_1',
    'log_rolling_6', 'hour_x_zone', 'recent_activity',
    'persistence',
    
    # missing ones
    'loc_avg_demand',
    'loc_std_demand'
]

assembler_correct = VectorAssembler(
    inputCols=correct_features,
    outputCol="features"
)

df_vec = assembler_correct.transform(df_filled)
df_scaled = scaler_model.transform(df_vec)

print(len(df_vec.first()["features"]))

32


In [109]:
df_vec.selectExpr("features").limit(3).show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                                                                                |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|(32,[0,1,10,11,12,13,14,15,24,27,30,31],[4051.0,-7401.0,7.0,5.0,1.0,0.9659258262890683,-0.25881904510252063,1.0,4.5716375605741976E-4,0.0032001462924019382,4.5716375605741976E-4,0.021377477442595494])|
|(32,[0,1,10,11,12,13,14,16,27,30,31],[4089.0,-7391.0,16.0,7.0,10.0,-0.8660254037844385,-0.5000000000000004,1.0,0.05851696077534973,0.003657310048459358,0.064752714295241])                

In [110]:
print(gbt_raw_model.numFeatures)

32


In [111]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(df_vec)
df_scaled = scaler_model.transform(df_vec)

from pyspark.ml.functions import vector_to_array

df_scaled.select(
    vector_to_array("features").alias("f"),
    vector_to_array("scaled_features").alias("sf")
).selectExpr(
    "size(f) as f_size",
    "size(sf) as sf_size"
).show(5)

+------+-------+
|f_size|sf_size|
+------+-------+
|    32|     32|
|    32|     32|
|    32|     32|
|    32|     32|
|    32|     32|
+------+-------+
only showing top 5 rows



In [112]:
df_pred_raw = gbt_raw_model.transform(df_scaled)
df_pred_raw.select("prediction").limit(10).show()

+------------------+
|        prediction|
+------------------+
|1.1304844027912053|
|1.0402278373687524|
|1.1304844027912053|
|1.1433095676282254|
|1.1304844027912053|
|1.3250534370959222|
|1.1304844027912053|
|1.1433095676282254|
|1.1304844027912053|
|1.0402278373687524|
+------------------+



In [113]:
df_pred_log = gbt_log_model.transform(df_scaled)
df_pred_log.select("prediction").limit(10).show()

+------------------+
|        prediction|
+------------------+
|0.6791169678495236|
|0.6717823069648304|
|0.6791169678495236|
| 0.677050449349216|
|0.6689257812702168|
| 0.677050449349216|
|0.6689257812702168|
|0.6779544792024773|
|0.6689257812702168|
|0.6722517078549628|
+------------------+



In [114]:
from pyspark.sql.functions import col, exp, when

df_final = df_pred_raw.select(
    "lat_bin", "lon_bin", "timestamp",
    col("prediction").alias("pred_raw")
).join(
    df_pred_log.select(
        "lat_bin", "lon_bin", "timestamp",
        exp(col("prediction")).alias("pred_log")
    ),
    on=["lat_bin", "lon_bin", "timestamp"],
    how="inner"
).withColumn(
    "final_prediction",
    when(col("pred_raw") > 5, col("pred_raw"))
    .otherwise(col("pred_log"))
)

In [115]:
from pyspark.sql.functions import date_format

df_model = df_model.withColumn(
    "year_month",
    date_format("timestamp", "yyyy-MM")
)

In [116]:
months = [row["year_month"] for row in df_model.select("year_month").distinct().collect()]
months.sort()

In [121]:
df_pred_raw.select("prediction").limit(5).show()
df_pred_log.select("prediction").limit(5).show()

+------------------+
|        prediction|
+------------------+
|1.1304844027912053|
|1.0402278373687524|
|1.1304844027912053|
|1.1433095676282254|
|1.1304844027912053|
+------------------+

+------------------+
|        prediction|
+------------------+
|0.6791169678495236|
|0.6717823069648304|
|0.6791169678495236|
| 0.677050449349216|
|0.6689257812702168|
+------------------+



In [122]:
from pyspark.sql.functions import col, exp, when

df_final_m = df_pred_raw.select(
    "lat_bin", "lon_bin", "timestamp",
    col("prediction").alias("pred_raw")
).join(
    df_pred_log.select(
        "lat_bin", "lon_bin", "timestamp",
        exp(col("prediction")).alias("pred_log")
    ),
    on=["lat_bin", "lon_bin", "timestamp"]
).withColumn(
    "final_prediction",
    when(col("pred_raw") > 5, col("pred_raw"))
    .otherwise(col("pred_log"))
)

In [123]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans

# ✅ separate column name
zone_assembler = VectorAssembler(
    inputCols=["lat_bin", "lon_bin"],
    outputCol="zone_features"
)

df_zone = df_model.select("lat_bin", "lon_bin").distinct()

df_zone_vec = zone_assembler.transform(df_zone)

# scale zone features
zone_scaler = StandardScaler(
    inputCol="zone_features",
    outputCol="zone_scaled",
    withStd=True,
    withMean=True
)

zone_scaler_model = zone_scaler.fit(df_zone_vec)
df_zone_scaled = zone_scaler_model.transform(df_zone_vec)

# KMeans
kmeans = KMeans(
    k=5,
    featuresCol="zone_scaled",
    predictionCol="zone_id",
    seed=42
)

kmeans_model = kmeans.fit(df_zone_scaled)

df_zone_clustered = kmeans_model.transform(df_zone_scaled).select(
    "lat_bin", "lon_bin", "zone_id"
)

df_model = df_model.join(
    df_zone_clustered,
    on=["lat_bin", "lon_bin"],
    how="left"
)

In [124]:
from pyspark.sql.functions import col, exp, when

months = ["2015-01"]  # test first

for m in months:
    print(f"\nProcessing: {m}")
    
    # Filter
    df_m = df_model.filter(col("year_month") == m)
    print("Rows:", df_m.count())
    
    # Fill nulls
    df_m = df_m.fillna(0, subset=correct_features)
    
    # Features
    df_vec = assembler_correct.transform(df_m)
    df_scaled = scaler_model.transform(df_vec)
    
    # Predictions
    df_pred_raw = gbt_raw_model.transform(df_scaled) \
        .withColumnRenamed("prediction", "pred_raw")
    
    df_pred_log = gbt_log_model.transform(df_scaled) \
        .withColumnRenamed("prediction", "pred_log")
    
    # FINAL JOIN (keeps ALL columns)
    df_final_m = df_m \
        .join(
            df_pred_raw.select(
                "lat_bin", "lon_bin", "timestamp", "pred_raw"
            ),
            on=["lat_bin", "lon_bin", "timestamp"],
            how="left"
        ) \
        .join(
            df_pred_log.select(
                "lat_bin", "lon_bin", "timestamp",
                exp(col("pred_log")).alias("pred_log")
            ),
            on=["lat_bin", "lon_bin", "timestamp"],
            how="left"
        ) \
        .withColumn(
            "final_prediction",
            when(col("pred_raw") > 5, col("pred_raw"))
            .otherwise(col("pred_log"))
        )
    
    # Save
    df_final_m.write.mode("overwrite").parquet("final_predictions_test")
    
    print(f"Finished: {m}")


Processing: 2015-01


Rows: 539412


Finished: 2015-01


In [125]:
df_check = spark.read.parquet("final_predictions_test")

print("Columns:", len(df_check.columns))
print("Has zone_id:", "zone_id" in df_check.columns)

df_check.select(
    "zone_id", "pred_raw", "pred_log", "final_prediction"
).show(5)

Columns: 44
Has zone_id: True
+-------+------------------+------------------+------------------+
|zone_id|          pred_raw|          pred_log|  final_prediction|
+-------+------------------+------------------+------------------+
|      4|1.1304844027912053|1.9721355040667277|1.9721355040667277|
|      4|1.1304844027912053|1.9721355040667277|1.9721355040667277|
|      4|1.1304844027912053| 1.952139169442089| 1.952139169442089|
|      4|1.1304844027912053| 1.952139169442089| 1.952139169442089|
|      4|1.1304844027912053| 1.952139169442089| 1.952139169442089|
+-------+------------------+------------------+------------------+
only showing top 5 rows



In [127]:
print("df_model cols:", len(df_model.columns))
print("df_final cols:", len(df_check.columns))

missing = set(df_model.columns) - set(df_check.columns)
extra = set(df_check.columns) - set(df_model.columns)

print("Missing from final:", missing)
print("Extra columns:", extra)

df_model cols: 41
df_final cols: 44
Missing from final: set()
Extra columns: {'final_prediction', 'pred_raw', 'pred_log'}


In [126]:
df_check.selectExpr(
    "avg(pred_raw)",
    "avg(pred_log)",
    "avg(final_prediction)"
).show()

+-----------------+------------------+---------------------+
|    avg(pred_raw)|     avg(pred_log)|avg(final_prediction)|
+-----------------+------------------+---------------------+
|18.94634244131283|19.641685598225123|   19.784337089957685|
+-----------------+------------------+---------------------+



In [128]:
df_check.show(5, truncate=False)

+-------+-------+-------------------+-----------+------+--------+------------+---------------+-----+-----+-----+---------+----+-----------+-----+-------------------+--------------------+---------------------+--------------------+------------+----------+--------+-------------+---------------------+---------+------+-------+---------+---------+--------------+---------------------+---------+-------------+---------------------+---------------+----------+-----------+----------+-----------+----------+-------+------------------+------------------+------------------+
|lat_bin|lon_bin|timestamp          |pickup_hour|demand|avg_fare|avg_distance|long_trip_ratio|lag_1|lag_2|lag_3|rolling_3|hour|day_of_week|month|hour_sin           |hour_cos            |loc_avg_demand       |loc_std_demand      |is_peak_hour|is_weekend|is_night|demand_vs_avg|demand_zscore        |rolling_6|lag_24|lag_168|trend_1_3|trend_1_6|is_active_zone|peak_intensity       |log_lag_1|log_rolling_6|hour_x_zone          |recent_ac

In [129]:
df_check.selectExpr(
    "count(*) as total_rows",
    "avg(demand) as avg_actual_demand",
    "avg(final_prediction) as avg_predicted",
    "avg(abs(demand - final_prediction)) as MAE"
).show()

+----------+-----------------+------------------+------------------+
|total_rows|avg_actual_demand|     avg_predicted|               MAE|
+----------+-----------------+------------------+------------------+
|    539412|18.13016210243747|19.784337089957685|3.7693220304806756|
+----------+-----------------+------------------+------------------+



In [130]:
from pyspark.sql.functions import sqrt, avg, pow

df_check.select(
    sqrt(avg(pow(col("demand") - col("final_prediction"), 2))).alias("RMSE")
).show()

+-----------------+
|             RMSE|
+-----------------+
|12.72623052077851|
+-----------------+



In [132]:
from pyspark.sql.functions import avg, abs

df_check.select(
    avg(abs(col("demand") - col("final_prediction"))).alias("MAE")
).show()

+------------------+
|               MAE|
+------------------+
|3.7693220304806756|
+------------------+



In [133]:
from pyspark.sql.functions import when

df_check.withColumn(
    "ape",
    when(col("demand") != 0,
         abs((col("demand") - col("final_prediction")) / col("demand"))
    )
).select(
    avg("ape").alias("MAPE")
).show()

+------------------+
|              MAPE|
+------------------+
|0.6514809735039788|
+------------------+



In [134]:
from pyspark.sql.functions import avg

mean_y = df_check.select(avg("demand")).first()[0]

df_check = df_check.withColumn(
    "ss_res", (col("demand") - col("final_prediction"))**2
).withColumn(
    "ss_tot", (col("demand") - mean_y)**2
)

r2 = 1 - (df_check.select(avg("ss_res")).first()[0] /
          df_check.select(avg("ss_tot")).first()[0])

print("R2:", r2)

R2: 0.9869343510469571


In [135]:
df_check.groupBy(
    when(col("demand") == 0, "zero").otherwise("non_zero").alias("segment")
).agg(
    avg("final_prediction").alias("avg_pred"),
    avg("demand").alias("avg_actual"),
    avg(abs(col("demand") - col("final_prediction"))).alias("MAE")
).show()

+--------+------------------+------------------+------------------+
| segment|          avg_pred|        avg_actual|               MAE|
+--------+------------------+------------------+------------------+
|    zero|1.9783737915761468|               0.0|1.9783737915761468|
|non_zero|101.86553917602001|101.70584258912601| 12.02516204653653|
+--------+------------------+------------------+------------------+



In [136]:
df_check.selectExpr("avg(demand)", "stddev(demand)").show()

+-----------------+------------------+
|      avg(demand)|    stddev(demand)|
+-----------------+------------------+
|18.13016210243747|111.33576449012558|
+-----------------+------------------+



In [131]:
df_check.groupBy(
    when(col("demand") > 5, "high").otherwise("low").alias("demand_type")
).agg(
    avg("final_prediction").alias("avg_pred"),
    avg("demand").alias("avg_actual")
).show()

+-----------+------------------+-------------------+
|demand_type|          avg_pred|         avg_actual|
+-----------+------------------+-------------------+
|        low|2.1183483232296316|0.19350720630984083|
|       high| 202.8101122050861| 203.96013226064613|
+-----------+------------------+-------------------+



In [ ]:
df_model = df_model.join(
    df_zone_clustered.select("lat_bin", "lon_bin", "zone_id"),
    on=["lat_bin", "lon_bin"],
    how="left"
)

In [ ]:
print("zone_id" in df_model.columns)

True


In [ ]:
from pyspark.sql.functions import date_format

df_model = df_model.withColumn(
    "year_month",
    date_format("timestamp", "yyyy-MM")
)

In [ ]:
months = [row["year_month"] for row in df_model.select("year_month").distinct().collect()]
months.sort()

print(months)

['2015-01', '2015-02', '2015-03', '2015-04', '2015-05', '2015-06', '2015-07', '2015-08', '2015-09', '2015-10', '2015-11', '2015-12', '2016-01', '2016-02', '2016-03']


In [ ]:
df_check = spark.read.parquet("final_predictions_test")
df_check.select("pred_raw", "pred_log", "final_prediction").show(5)
df_check.count()

+------------------+------------------+------------------+
|          pred_raw|          pred_log|  final_prediction|
+------------------+------------------+------------------+
|1.1304844027912053|1.9721355040667277|1.9721355040667277|
|1.1304844027912053|1.9721355040667277|1.9721355040667277|
|1.1304844027912053| 1.952139169442089| 1.952139169442089|
|1.1304844027912053| 1.952139169442089| 1.952139169442089|
|1.1304844027912053| 1.952139169442089| 1.952139169442089|
+------------------+------------------+------------------+
only showing top 5 rows



539412

In [ ]:
df_check.columns

['lat_bin', 'lon_bin', 'timestamp', 'pred_raw', 'pred_log', 'final_prediction']

In [137]:
from pyspark.sql.functions import col, exp, when

months = [
    '2015-01','2015-02','2015-03','2015-04','2015-05','2015-06',
    '2015-07','2015-08','2015-09','2015-10','2015-11','2015-12',
    '2016-01','2016-02','2016-03'
]

for m in months:
    print(f"\nProcessing: {m}")
    
    # STEP 1: Filter
    df_m = df_model.filter(col("year_month") == m)
    print("Rows:", df_m.count())
    
    # STEP 2: Fill nulls
    df_m = df_m.fillna(0, subset=correct_features)
    
    # STEP 3: Features
    df_vec = assembler_correct.transform(df_m)
    df_scaled = scaler_model.transform(df_vec)
    
    # STEP 4: Predictions
    df_pred_raw = gbt_raw_model.transform(df_scaled) \
        .withColumnRenamed("prediction", "pred_raw")
    
    df_pred_log = gbt_log_model.transform(df_scaled) \
        .withColumnRenamed("prediction", "pred_log")
    
    # STEP 5: Join back (ALL columns preserved)
    df_final_m = df_m \
        .join(
            df_pred_raw.select(
                "lat_bin", "lon_bin", "timestamp", "pred_raw"
            ),
            ["lat_bin", "lon_bin", "timestamp"],
            "left"
        ) \
        .join(
            df_pred_log.select(
                "lat_bin", "lon_bin", "timestamp",
                exp(col("pred_log")).alias("pred_log")
            ),
            ["lat_bin", "lon_bin", "timestamp"],
            "left"
        ) \
        .withColumn(
            "final_prediction",
            when(col("pred_raw") > col("pred_log"), col("pred_raw"))
            .otherwise(col("pred_log"))
        ) \
        .withColumn(
            "final_prediction",
            when(col("final_prediction") < 2, 0)   # 🔥 threshold fix
            .otherwise(col("final_prediction"))
        )
    
    # STEP 6: Save per month (safe)
    df_final_m.write.mode("overwrite") \
        .parquet(f"final_predictions/year_month={m}")
    
    print(f"Finished: {m}")


Processing: 2015-01


Rows: 539412


Finished: 2015-01

Processing: 2015-02


Rows: 637056


Finished: 2015-02

Processing: 2015-03


...

Finished: 2016-01

Processing: 2016-02


Rows: 659808


Finished: 2016-02

Processing: 2016-03


Rows: 705312


Finished: 2016-03


In [138]:
from pyspark.sql.functions import round, col

df_all = spark.read.parquet("final_predictions")

df_all = df_all.withColumn(
    "pred_raw", round(col("pred_raw"))
).withColumn(
    "pred_log", round(col("pred_log"))
).withColumn(
    "final_prediction", round(col("final_prediction"))
)

26/04/06 16:19:35 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `year_month` already exists. Consider to choose another name or rename the existing column.


In [139]:
from pyspark.sql.functions import round, col

df_all = spark.read.parquet("final_predictions")

df_all = df_all \
    .withColumn("pred_raw", round(col("pred_raw")).cast("int")) \
    .withColumn("pred_log", round(col("pred_log")).cast("int")) \
    .withColumn("final_prediction", round(col("final_prediction")).cast("int"))

26/04/06 16:20:02 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `year_month` already exists. Consider to choose another name or rename the existing column.


In [140]:
df_all.show(5, truncate=False)

+-------+-------+-------------------+-----------+------+--------+------------+---------------+-----+-----+-----+---------+----+-----------+-----+-------------------+------------------+--------------------+-----------------+------------+----------+--------+-------------+---------------------+---------+------+-------+---------+---------+--------------+--------------+---------+-------------+--------------------+---------------+----------+-----------+----------+-----------+----------+-------+--------+--------+----------------+
|lat_bin|lon_bin|timestamp          |pickup_hour|demand|avg_fare|avg_distance|long_trip_ratio|lag_1|lag_2|lag_3|rolling_3|hour|day_of_week|month|hour_sin           |hour_cos          |loc_avg_demand      |loc_std_demand   |is_peak_hour|is_weekend|is_night|demand_vs_avg|demand_zscore        |rolling_6|lag_24|lag_168|trend_1_3|trend_1_6|is_active_zone|peak_intensity|log_lag_1|log_rolling_6|hour_x_zone         |recent_activity|log_demand|persistence|has_demand|pickup_da

In [141]:
df_all.select(
    "year_month",
    "zone_id",
    "timestamp",
    "demand",
    "final_prediction"
).show(10, truncate=False)

+----------+-------+-------------------+------+----------------+
|year_month|zone_id|timestamp          |demand|final_prediction|
+----------+-------+-------------------+------+----------------+
|2016-01   |0      |2016-01-01 00:00:00|0     |0               |
|2016-01   |0      |2016-01-01 01:00:00|0     |2               |
|2016-01   |0      |2016-01-01 02:00:00|0     |0               |
|2016-01   |0      |2016-01-01 03:00:00|0     |0               |
|2016-01   |0      |2016-01-01 04:00:00|0     |0               |
|2016-01   |0      |2016-01-01 05:00:00|0     |0               |
|2016-01   |0      |2016-01-01 06:00:00|0     |0               |
|2016-01   |0      |2016-01-01 07:00:00|0     |0               |
|2016-01   |0      |2016-01-01 08:00:00|0     |0               |
|2016-01   |0      |2016-01-01 09:00:00|0     |0               |
+----------+-------+-------------------+------+----------------+
only showing top 10 rows



In [142]:
df_all.orderBy("timestamp").select(
    "year_month",
    "zone_id",
    "timestamp",
    "demand",
    "final_prediction"
).show(10, truncate=False)

+----------+-------+-------------------+------+----------------+
|year_month|zone_id|timestamp          |demand|final_prediction|
+----------+-------+-------------------+------+----------------+
|2015-01   |0      |2015-01-08 07:00:00|0     |0               |
|2015-01   |4      |2015-01-08 07:00:00|2     |3               |
|2015-01   |4      |2015-01-08 07:00:00|0     |0               |
|2015-01   |4      |2015-01-08 07:00:00|3     |4               |
|2015-01   |4      |2015-01-08 07:00:00|0     |0               |
|2015-01   |4      |2015-01-08 07:00:00|7     |6               |
|2015-01   |0      |2015-01-08 07:00:00|0     |0               |
|2015-01   |4      |2015-01-08 07:00:00|1     |2               |
|2015-01   |4      |2015-01-08 07:00:00|0     |0               |
|2015-01   |4      |2015-01-08 07:00:00|1     |2               |
+----------+-------+-------------------+------+----------------+
only showing top 10 rows



In [143]:
df_all.sample(0.0001).select(
    "zone_id",
    "demand",
    "final_prediction"
).show(10, truncate=False)

+-------+------+----------------+
|zone_id|demand|final_prediction|
+-------+------+----------------+
|2      |0     |0               |
|2      |0     |0               |
|3      |0     |0               |
|1      |0     |0               |
|0      |0     |0               |
|0      |0     |0               |
|1      |0     |0               |
|1      |0     |0               |
|0      |0     |0               |
|3      |0     |0               |
+-------+------+----------------+
only showing top 10 rows



In [144]:
df_all.select(
    "demand",
    "final_prediction"
).show(20, truncate=False)

+------+----------------+
|demand|final_prediction|
+------+----------------+
|0     |0               |
|0     |2               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
|0     |0               |
+------+----------------+
only showing top 20 rows



In [145]:
df_all.selectExpr(
    "count(*) as total_rows",
    "sum(case when demand = 0 then 1 else 0 end) as zero_rows",
    "sum(case when demand > 0 then 1 else 0 end) as non_zero_rows"
).show()

+----------+---------+-------------+
|total_rows|zero_rows|non_zero_rows|
+----------+---------+-------------+
|  10209012|  9741937|       467075|
+----------+---------+-------------+



In [146]:
df_non_zero = df_all.filter(col("demand") > 0)

In [147]:
df_non_zero.select(
    "zone_id", "timestamp", "demand", "final_prediction"
).show(20, truncate=False)

+-------+-------------------+------+----------------+
|zone_id|timestamp          |demand|final_prediction|
+-------+-------------------+------+----------------+
|4      |2016-01-01 20:00:00|1     |0               |
|4      |2016-01-01 01:00:00|1     |0               |
|4      |2016-01-01 17:00:00|1     |2               |
|4      |2016-01-01 01:00:00|1     |2               |
|4      |2016-01-01 02:00:00|2     |3               |
|4      |2016-01-01 04:00:00|1     |0               |
|4      |2016-01-01 03:00:00|1     |0               |
|4      |2016-01-01 05:00:00|1     |2               |
|4      |2016-01-01 03:00:00|1     |0               |
|4      |2016-01-01 01:00:00|1     |0               |
|4      |2016-01-01 04:00:00|2     |2               |
|4      |2016-01-01 04:00:00|1     |0               |
|4      |2016-01-01 04:00:00|1     |0               |
|4      |2016-01-01 04:00:00|2     |3               |
|4      |2016-01-01 02:00:00|1     |2               |
|4      |2016-01-01 10:00:00

In [148]:
df_non_zero.select("demand").describe().show()

+-------+------------------+
|summary|            demand|
+-------+------------------+
|  count|            467075|
|   mean| 92.87449124872879|
| stddev|224.59348119916194|
|    min|                 1|
|    max|              2658|
+-------+------------------+



In [149]:
df_non_zero.groupBy("demand").count().orderBy("demand").show(20)

+------+------+
|demand| count|
+------+------+
|     1|130142|
|     2| 48482|
|     3| 29232|
|     4| 20296|
|     5| 14854|
|     6| 11859|
|     7|  9706|
|     8|  8441|
|     9|  7047|
|    10|  6236|
|    11|  5377|
|    12|  4800|
|    13|  4210|
|    14|  3918|
|    15|  3587|
|    16|  3178|
|    17|  2987|
|    18|  2720|
|    19|  2527|
|    20|  2344|
+------+------+
only showing top 20 rows



In [150]:
df_all.groupBy("zone_id").agg(
    {"demand": "sum"}
).orderBy("sum(demand)", ascending=False).show(10)

+-------+-----------+
|zone_id|sum(demand)|
+-------+-----------+
|      2|   40063969|
|      1|    1935974|
|      3|     902512|
|      4|     305918|
|      0|     170980|
+-------+-----------+



In [151]:
df_all.groupBy("hour").agg(
    {"demand": "avg"}
).orderBy("hour").show()

+----+------------------+
|hour|       avg(demand)|
+----+------------------+
|   0|3.6083672393007835|
|   1| 2.627700704490657|
|   2|1.9266100625376732|
|   3| 1.435891821127185|
|   4|1.0656928119349005|
|   5|1.0019613660337552|
|   6|2.2132944356540083|
|   7| 3.834848655709359|
|   8|  4.70425370960315|
|   9| 4.736651067068873|
|  10|  4.57702301410542|
|  11| 4.743062407788522|
|  12| 5.024148835198707|
|  13|5.0111546521571615|
|  14| 5.228200501818387|
|  15| 5.135566143234379|
|  16| 4.629293413398739|
|  17| 5.439946247169049|
|  18| 6.455435426122748|
|  19|6.4397888415889035|
+----+------------------+
only showing top 20 rows



In [152]:
df_clean = df_all.filter(col("final_prediction") > 0)

In [153]:
df_clean.select("zone_id", "final_prediction").show(20)

+-------+----------------+
|zone_id|final_prediction|
+-------+----------------+
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
|      0|               2|
+-------+----------------+
only showing top 20 rows



In [154]:
df_active = df_all.filter(col("demand") > 0)

In [155]:
df_all

DataFrame[lat_bin: bigint, lon_bin: bigint, timestamp: timestamp, pickup_hour: int, demand: bigint, avg_fare: double, avg_distance: double, long_trip_ratio: double, lag_1: bigint, lag_2: bigint, lag_3: bigint, rolling_3: double, hour: int, day_of_week: int, month: int, hour_sin: double, hour_cos: double, loc_avg_demand: double, loc_std_demand: double, is_peak_hour: int, is_weekend: int, is_night: int, demand_vs_avg: double, demand_zscore: double, rolling_6: double, lag_24: bigint, lag_168: bigint, trend_1_3: bigint, trend_1_6: double, is_active_zone: int, peak_intensity: double, log_lag_1: double, log_rolling_6: double, hour_x_zone: double, recent_activity: double, log_demand: double, persistence: int, has_demand: int, pickup_date: date, year_month: string, zone_id: int, pred_raw: int, pred_log: int, final_prediction: int]

In [156]:
df_active.select("demand").describe().show()

+-------+------------------+
|summary|            demand|
+-------+------------------+
|  count|            467075|
|   mean| 92.87449124872879|
| stddev|224.59348119916194|
|    min|                 1|
|    max|              2658|
+-------+------------------+



In [157]:
df_active.groupBy("zone_id").agg(
    {"demand": "sum"}
).orderBy("sum(demand)", ascending=False).show(10)

+-------+-----------+
|zone_id|sum(demand)|
+-------+-----------+
|      2|   40063969|
|      1|    1935974|
|      3|     902512|
|      4|     305918|
|      0|     170980|
+-------+-----------+



In [158]:
df_active.groupBy("hour").agg(
    {"demand": "avg"}
).orderBy("hour").show()

+----+------------------+
|hour|       avg(demand)|
+----+------------------+
|   0| 70.23318056828597|
|   1| 54.10884848484849|
|   2| 44.25544918600249|
|   3| 35.40371552975326|
|   4|25.560738690913198|
|   5| 24.03077704992094|
|   6| 48.39597384544097|
|   7| 78.24326526699262|
|   8| 96.55583952165108|
|   9|104.06012903225806|
|  10|105.75502117033982|
|  11| 111.3811100077237|
|  12| 116.4527880636027|
|  13|  115.254120062679|
|  14|116.89221556886227|
|  15|113.63921813266792|
|  16|103.81812434141202|
|  17|119.62204887120939|
|  18|139.86404357121043|
|  19|138.31410838631547|
+----+------------------+
only showing top 20 rows



In [ ]:
df_active.select(
    "demand",
    "final_prediction"
).show(20)

+------+----------------+
|demand|final_prediction|
+------+----------------+
|     1|               0|
|     1|               0|
|     1|               2|
|     1|               2|
|     2|               3|
|     1|               0|
|     1|               0|
|     1|               2|
|     1|               0|
|     1|               0|
|     2|               2|
|     1|               0|
|     1|               0|
|     2|               3|
|     1|               2|
|     1|               0|
|     1|               2|
|     2|               2|
|     1|               2|
|     1|               0|
+------+----------------+
only showing top 20 rows



26/04/06 22:12:44 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 143126 ms exceeds timeout 120000 ms
26/04/06 22:12:44 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/06 22:12:44 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$